# Heart Disease — Decision Tree Classification
## Project 8

### Objective
Build a complete binary classification workflow using a **Decision Tree Classifier**.

Topics covered:
- EDA
- missing-value handling
- categorical encoding
- train/test split
- Decision Tree hyperparameters
- classification metrics
- confusion matrix
- ROC-AUC
- feature importance
- tree visualization
- stratified 5-fold cross-validation

> **Dataset note:** The included 920-row CSV is a reproducible educational practice dataset modeled on the common Heart Disease feature schema. It is not a clinical dataset and is not intended for diagnosis or treatment.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss,confusion_matrix,classification_report,RocCurveDisplay

BASE=Path.cwd()
if not (BASE/'dataset').exists(): BASE=BASE.parent
DATA=BASE/'dataset'/'heart_disease.csv'; OUT=BASE/'outputs'; OUT.mkdir(exist_ok=True)
df=pd.read_csv(DATA)
display(df.head()); print('Shape:',df.shape)

## 1. Data Quality and EDA

In [ ]:
display(df.info())
display(df.describe().T)
display(pd.DataFrame({'Missing_Count':df.isna().sum(),'Missing_Percent':df.isna().mean()*100}))
display(df['target'].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(7,5)); plt.hist(df.age,bins=20); plt.title('Age Distribution'); plt.xlabel('Age'); plt.ylabel('Frequency'); plt.show()
plt.figure(figsize=(7,5)); df.groupby('target').age.mean().plot(kind='bar'); plt.title('Average Age by Target'); plt.xlabel('Target'); plt.ylabel('Average Age'); plt.show()
plt.figure(figsize=(7,5)); df.groupby('cp').target.mean().plot(kind='bar'); plt.title('Disease Rate by Chest Pain Category'); plt.xlabel('cp'); plt.ylabel('Disease Rate'); plt.show()

## 2. Feature Preparation

Numeric features:
`age, trestbps, chol, thalach, oldpeak`

Categorical features:
`sex, cp, fbs, restecg, exang, slope, ca, thal`

Target:
`target` — 0 = no disease, 1 = disease.

The preprocessing is kept inside the pipeline to avoid inconsistent transformations.

In [ ]:
num=['age','trestbps','chol','thalach','oldpeak']
cat=['sex','cp','fbs','restecg','exang','slope','ca','thal']
X=df[num+cat]; y=df['target']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)
pre=ColumnTransformer([
 ('num',Pipeline([('imputer',SimpleImputer(strategy='median'))]),num),
 ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                  ('onehot',OneHotEncoder(handle_unknown='ignore',drop='first'))]),cat)
])

## 3. Decision Tree Model

The tree uses:

- `max_depth=5` — limits tree complexity
- `min_samples_leaf=8` — prevents extremely small leaves
- `class_weight='balanced'` — accounts for class imbalance
- `random_state=42` — reproducibility

In [ ]:
model=Pipeline([
 ('preprocessor',pre),
 ('classifier',DecisionTreeClassifier(max_depth=5,min_samples_leaf=8,class_weight='balanced',random_state=42))
])
model.fit(X_train,y_train)
pred=model.predict(X_test)
proba=model.predict_proba(X_test)[:,1]

## 4. Classification Metrics

In [ ]:
metrics=pd.DataFrame([{
 'Accuracy':accuracy_score(y_test,pred),
 'Precision':precision_score(y_test,pred),
 'Recall':recall_score(y_test,pred),
 'F1':f1_score(y_test,pred),
 'ROC_AUC':roc_auc_score(y_test,proba),
 'Log_Loss':log_loss(y_test,proba)
}])
display(metrics)
metrics.to_csv(OUT/'decision_tree_metrics.csv',index=False)
display(pd.DataFrame(classification_report(y_test,pred,output_dict=True)).T)

## 5. Confusion Matrix

In [ ]:
cm=confusion_matrix(y_test,pred)
display(pd.DataFrame(cm,index=['Actual_0','Actual_1'],columns=['Predicted_0','Predicted_1']))
plt.figure(figsize=(6,5)); plt.imshow(cm); plt.title('Confusion Matrix')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.xticks([0,1]); plt.yticks([0,1])
for i in range(2):
    for j in range(2): plt.text(j,i,cm[i,j],ha='center',va='center')
plt.show()
pd.DataFrame(cm,index=['Actual_0','Actual_1'],columns=['Predicted_0','Predicted_1']).to_csv(OUT/'confusion_matrix.csv')

## 6. ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(y_test,proba)
plt.title('Decision Tree ROC Curve'); plt.show()

## 7. Feature Importance

In [ ]:
feature_names=model.named_steps['preprocessor'].get_feature_names_out()
importance=pd.DataFrame({
 'Feature':feature_names,
 'Importance':model.named_steps['classifier'].feature_importances_
}).sort_values('Importance',ascending=False)
display(importance.head(20))
importance.to_csv(OUT/'feature_importance.csv',index=False)

## 8. Decision Tree Visualization

In [ ]:
plt.figure(figsize=(14,8))
plot_tree(model.named_steps['classifier'],
          feature_names=feature_names,
          class_names=['No Disease','Disease'],
          filled=False,max_depth=3,fontsize=7)
plt.title('Decision Tree — First 3 Levels')
plt.show()

## 9. Test Predictions

In [ ]:
predictions=df.loc[X_test.index].copy()
predictions['Predicted_Target']=pred
predictions['Probability_Target_1']=proba
display(predictions.head())
predictions.to_csv(OUT/'test_predictions.csv',index=False)

## 10. Stratified 5-Fold Cross-Validation

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
r=cross_validate(model,X,y,cv=cv,scoring=['accuracy','precision','recall','f1','roc_auc'],n_jobs=-1)
cv_summary=pd.DataFrame({
 'Metric':['Accuracy','Precision','Recall','F1','ROC_AUC'],
 'Mean':[r['test_accuracy'].mean(),r['test_precision'].mean(),r['test_recall'].mean(),r['test_f1'].mean(),r['test_roc_auc'].mean()],
 'Std':[r['test_accuracy'].std(),r['test_precision'].std(),r['test_recall'].std(),r['test_f1'].std(),r['test_roc_auc'].std()]
})
display(cv_summary); cv_summary.to_csv(OUT/'cross_validation_results.csv',index=False)

## 11. Conclusion

The completed workflow is:

**Heart Disease data → EDA → missing-value handling → categorical encoding →
stratified split → Decision Tree → classification metrics → confusion matrix →
ROC-AUC → feature importance → cross-validation.**

The model is suitable as an educational machine-learning exercise only; it must not be
used as a medical diagnostic system.